In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt

from sklearn import linear_model
import lightgbm as lgbm

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [2]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv',index_col='ROW_ID')
y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)
print(total_data['target'])

ROW_ID
0         0.009210
1         0.000405
2         0.005967
3        -0.000542
4        -0.002579
            ...   
527068   -0.001769
527069    0.005640
527070    0.004016
527071    0.001617
527072    0.000556
Name: target, Length: 527073, dtype: float64


In [5]:
df = total_data.copy()

df["TS_RET1_MEDIAN"] = df.groupby("TS")["RET_1"].transform("median")

df["RET1_REL"] = df["RET_1"] - df["TS_RET1_MEDIAN"]

print(df[["RET1_REL", "target"]].corr())

          RET1_REL    target
RET1_REL  1.000000  0.081601
target    0.081601  1.000000


In [7]:
df = total_data.copy()

df["RET1_RANK"] = df.groupby("TS")["RET_1"].rank(pct=True)

print(df[["RET1_RANK", "target"]].corr())

           RET1_RANK    target
RET1_RANK   1.000000  0.060889
target      0.060889  1.000000


In [8]:
df = total_data.copy()

df["GROUP_RET1_MEDIAN"] = (
    df.groupby(["TS", "GROUP"])["RET_1"]
      .transform("median")
)

df["RET1_GROUP_REL"] = df["RET_1"] - df["GROUP_RET1_MEDIAN"]

print(df[["RET1_GROUP_REL", "target"]].corr())

                RET1_GROUP_REL    target
RET1_GROUP_REL        1.000000  0.074245
target                0.074245  1.000000


In [9]:
df = total_data.copy()

g = df.groupby("TS")["RET_1"]

df["TS_RET1_MEAN"] = g.transform("mean")
df["TS_RET1_STD"] = g.transform("std")

df["RET1_CS_Z"] = (
    (df["RET_1"] - df["TS_RET1_MEAN"])
    / df["TS_RET1_STD"]
)

print(df[["RET1_CS_Z", "target"]].corr())

           RET1_CS_Z    target
RET1_CS_Z   1.000000  0.065416
target      0.065416  1.000000


In [10]:
df = total_data.copy()

df["RET5"] = df[[f"RET_{i}" for i in range(1, 6)]].mean(axis=1)

df["RET5_CS_MEDIAN"] = df.groupby("TS")["RET5"].transform("median")

df["RET5_REL"] = df["RET5"] - df["RET5_CS_MEDIAN"]

print(df[["RET5_REL", "target"]].corr())

          RET5_REL    target
RET5_REL  1.000000  0.014664
target    0.014664  1.000000


In [11]:
for i in range(1, 11):
    med = total_data.groupby("TS")[f"RET_{i}"].transform("median")
    rel = total_data[f"RET_{i}"] - med
    corr = rel.corr(total_data["target"])
    print(f"RET_{i}: {corr:.5f}")

RET_1: 0.08160
RET_2: -0.01078
RET_3: -0.01176
RET_4: -0.02131
RET_5: -0.00175
RET_6: 0.01077
RET_7: 0.01967
RET_8: 0.01990
RET_9: 0.01824
RET_10: -0.00144


In [12]:
df = total_data.copy()

df["RET1_CS_MEDIAN"] = df.groupby("TS")["RET_1"].transform("median")
df["RET1_REL"] = df["RET_1"] - df["RET1_CS_MEDIAN"]

print(
    df.groupby("GROUP")["RET1_REL"]
      .corr(df["target"])
)

GROUP
1    0.110684
2    0.081716
3    0.072517
4    0.071563
Name: RET1_REL, dtype: float64


In [13]:
df = total_data.copy()

df["RET1_CS_MEDIAN"] = df.groupby("TS")["RET_1"].transform("median")
df["RET1_REL"] = df["RET_1"] - df["RET1_CS_MEDIAN"]

df["RET1_REL_DECILE"] = (
    df.groupby("TS")["RET1_REL"]
      .transform(lambda x: x.rank(pct=True))
)

df["DECILE"] = (df["RET1_REL_DECILE"] * 10).clip(upper=10).astype(int)

print(
    df.groupby("DECILE")["target"]
      .agg(["mean", "count"])
)

            mean  count
DECILE                 
0      -0.000326  51133
1      -0.000153  53206
2      -0.000098  51976
3      -0.000035  53205
4       0.000023  51740
5       0.000051  53379
6       0.000092  53482
7       0.000170  51699
8       0.000227  53482
9       0.000365  51249
10      0.000639   2522


In [15]:
df["TARGET_POS"] = (df["target"] > 0).astype(int)

print(
    df.groupby("DECILE")["TARGET_POS"]
      .agg(["mean", "count"])
)

            mean  count
DECILE                 
0       0.474449  51133
1       0.486242  53206
2       0.489303  51976
3       0.494220  53205
4       0.505354  51740
5       0.509507  53379
6       0.515089  53482
7       0.525020  51699
8       0.532216  53482
9       0.538137  51249
10      0.549960   2522


In [16]:
print(
    df.groupby(["GROUP", "DECILE"])["TARGET_POS"]
      .mean()
      .unstack(0)
)

GROUP          1         2         3         4
DECILE                                        
0       0.458009  0.463456  0.490100  0.471698
1       0.472860  0.490814  0.490340  0.485846
2       0.486527  0.487914  0.493183  0.488010
3       0.479959  0.498618  0.508273  0.485917
4       0.500941  0.507156  0.507736  0.504647
5       0.502472  0.516315  0.504051  0.514229
6       0.514695  0.516273  0.511649  0.518042
7       0.527066  0.521817  0.522549  0.529943
8       0.529708  0.536582  0.530313  0.531793
9       0.532613  0.550758  0.535877  0.527413
10      0.515789  0.564057  0.561550  0.511364


In [18]:
print("RET_1:", df[["RET_1", "target"]].corr().iloc[0, 1])
print("RET1_REL:", df[["RET1_REL", "target"]].corr().iloc[0, 1])

RET_1: 0.08485279746760785
RET1_REL: 0.08160107209573225


In [20]:
df["RET1_RANK"] = (
    df.groupby("TS")["RET_1"]
      .rank(pct=True)
)

df["RET1_RANK_DECILE"] = (
    df["RET1_RANK"] * 10
).clip(upper=10).astype(int)

print(
    df.groupby("RET1_RANK_DECILE")["target"]
      .agg(["mean", "count"])
)

                      mean  count
RET1_RANK_DECILE                 
0                -0.000326  51133
1                -0.000153  53206
2                -0.000098  51976
3                -0.000035  53205
4                 0.000023  51740
5                 0.000051  53379
6                 0.000092  53482
7                 0.000170  51699
8                 0.000227  53482
9                 0.000365  51249
10                0.000639   2522


In [22]:
print(
    df.loc[df["RET_1"].isna(), "target"]
      .agg(["mean", "count"])
)

print(
    df.loc[df["RET_1"].notna(), "target"]
      .agg(["mean", "count"])
)

mean     NaN
count    0.0
Name: target, dtype: float64
mean          0.000035
count    527073.000000
Name: target, dtype: float64


In [24]:
df["RET2_RANK"] = (
    df.groupby("TS")["RET_2"]
      .rank(pct=True)
)

print(df[["RET2_RANK", "target"]].corr())

           RET2_RANK    target
RET2_RANK   1.000000 -0.005828
target     -0.005828  1.000000


In [25]:
df["RET3_RANK"] = (
    df.groupby("TS")["RET_3"]
      .rank(pct=True)
)

print(df[["RET3_RANK", "target"]].corr())

           RET3_RANK    target
RET3_RANK   1.000000 -0.007601
target     -0.007601  1.000000


In [26]:
df["SV1_RANK"] = (
    df.groupby("TS")["SIGNED_VOLUME_1"]
      .rank(pct=True)
)

print(df[["SV1_RANK", "target"]].corr())

          SV1_RANK    target
SV1_RANK  1.000000  0.004281
target    0.004281  1.000000


In [28]:
df["ABS_SV1"] = df["SIGNED_VOLUME_1"].abs()

df["ABS_SV1_RANK"] = (
    df.groupby("TS")["ABS_SV1"]
      .rank(pct=True)
)

print(df[["ABS_SV1_RANK", "target"]].corr())

              ABS_SV1_RANK    target
ABS_SV1_RANK      1.000000  0.006518
target            0.006518  1.000000


In [30]:
df["ABS_RET1"] = df["RET_1"].abs()

print(
    df.groupby(
        pd.qcut(df["ABS_RET1"], 10, duplicates="drop")
    )[["RET_1", "target"]]
    .corr()
    .iloc[0::2]["target"]
)

ABS_RET1                         
(-0.00099999822, 0.000268]  RET_1    0.005926
(0.000268, 0.000544]        RET_1    0.010707
(0.000544, 0.000839]        RET_1    0.017963
(0.000839, 0.00116]         RET_1    0.017951
(0.00116, 0.00152]          RET_1    0.037944
(0.00152, 0.00195]          RET_1    0.045977
(0.00195, 0.00249]          RET_1    0.049903
(0.00249, 0.00325]          RET_1    0.067870
(0.00325, 0.00461]          RET_1    0.078540
(0.00461, 0.129]            RET_1    0.147184
Name: target, dtype: float64


In [31]:
df["RET1_SIGN"] = np.sign(df["RET_1"])
df["RET1_ABS"] = df["RET_1"].abs()

df["RET1_SIGN_ABS"] = df["RET1_SIGN"] * df["RET1_ABS"]

print(df[["RET1_SIGN_ABS", "target"]].corr())

               RET1_SIGN_ABS    target
RET1_SIGN_ABS       1.000000  0.084853
target              0.084853  1.000000


In [32]:
df["TARGET_POS"] = (df["target"] > 0).astype(int)
df["ABS_RET1_DECILE"] = pd.qcut(df["RET_1"].abs(), 10, labels=False)

print(
    df.groupby("ABS_RET1_DECILE")["TARGET_POS"].mean()
)

ABS_RET1_DECILE
0    0.504705
1    0.506005
2    0.507295
3    0.505113
4    0.505616
5    0.510767
6    0.505739
7    0.507333
8    0.506916
9    0.512351
Name: TARGET_POS, dtype: float64


In [33]:
df["RET1_RANK"] = (
    df.groupby("TS")["RET_1"].rank(pct=True)
)

df["RET1_RANK_X_RET1"] = df["RET1_RANK"] * df["RET_1"]

print(
    df[["RET1_RANK_X_RET1", "target"]].corr()
)

                  RET1_RANK_X_RET1    target
RET1_RANK_X_RET1          1.000000  0.072282
target                    0.072282  1.000000


In [34]:
df["RET1_CS_Z"] = (
    df["RET_1"] - df.groupby("TS")["RET_1"].transform("mean")
) / df.groupby("TS")["RET_1"].transform("std")

print(df[["RET1_CS_Z", "target"]].corr())

           RET1_CS_Z    target
RET1_CS_Z   1.000000  0.065416
target      0.065416  1.000000


In [35]:
df["RET1_MINUS_RET2"] = df["RET_1"] - df["RET_2"]

print(
    df[["RET1_MINUS_RET2", "target"]].corr()
)

                 RET1_MINUS_RET2   target
RET1_MINUS_RET2          1.00000  0.06849
target                   0.06849  1.00000


In [36]:
df["RET1_RET2_SAME_SIGN"] = (
    np.sign(df["RET_1"]) == np.sign(df["RET_2"])
).astype(int)

print(
    df.groupby("RET1_RET2_SAME_SIGN")["target"]
      .agg(["mean", "count"])
)

                         mean   count
RET1_RET2_SAME_SIGN                  
0                    0.000033  255947
1                    0.000036  271126


In [38]:
df["RET1_RANK"] = df.groupby("TS")["RET_1"].rank(pct=True)

df["RET1_SIGN"] = np.sign(df["RET_1"])

df["RET1_SIGN_RANK"] = df["RET1_SIGN"] * df["RET1_RANK"]

print(
    df[["RET1_SIGN_RANK", "target"]].corr()
)
df["RET1_SIGN"] = np.sign(df["RET_1"])
df["RET1_HIGH_RANK"] = (
    df.groupby("TS")["RET_1"].rank(pct=True) > 0.5
)

print(
    df.groupby(["RET1_SIGN", "RET1_HIGH_RANK"])["target"]
      .agg(["mean", "count"])
)

                RET1_SIGN_RANK    target
RET1_SIGN_RANK        1.000000  0.050056
target                0.050056  1.000000
                              mean   count
RET1_SIGN RET1_HIGH_RANK                  
-1.0      False          -0.000138  241755
          True           -0.000019   15603
 1.0      False           0.000132   21536
          True            0.000198  248179


In [39]:
df["RET_RECENT_3"] = df[["RET_1", "RET_2", "RET_3"]].mean(axis=1)

df["RET_OLD_7"] = df[[f"RET_{i}" for i in range(4, 11)]].mean(axis=1)

df["RET_ACCEL"] = df["RET_RECENT_3"] - df["RET_OLD_7"]

print(
    df[["RET_ACCEL", "target"]].corr()
)

           RET_ACCEL    target
RET_ACCEL   1.000000  0.020761
target      0.020761  1.000000


In [40]:
alloc_corr = (
    df.groupby("ALLOCATION")
      .apply(lambda g: g["RET_1"].corr(g["target"]))
)

print(alloc_corr.describe())

count    278.000000
mean       0.083332
std        0.047320
min       -0.025727
25%        0.048294
50%        0.076926
75%        0.118179
max        0.257227
dtype: float64


In [42]:
df["ROW_HALF"] = df.groupby("ALLOCATION").cumcount() % 2

corrs = (
    df.groupby(["ALLOCATION", "ROW_HALF"])
      .apply(lambda g: g["RET_1"].corr(g["target"]))
      .unstack()
)

print(corrs.corr())

ROW_HALF         0         1
ROW_HALF                    
0         1.000000  0.277913
1         0.277913  1.000000


In [43]:
def slope(g):
    return g["RET_1"].cov(g["target"]) / g["RET_1"].var()

slopes = (
    df.groupby(["ALLOCATION", "ROW_HALF"])
      .apply(slope)
      .unstack()
)

print(slopes.corr())
print(slopes.describe())

ROW_HALF         0         1
ROW_HALF                    
0         1.000000  0.313046
1         0.313046  1.000000
ROW_HALF           0           1
count     278.000000  278.000000
mean        0.088445    0.082363
std         0.060526    0.059681
min        -0.070215   -0.036256
25%         0.046384    0.034192
50%         0.086795    0.078058
75%         0.123098    0.127265
max         0.393045    0.276828


In [44]:
# slope estimated from half 0
train_half = df[df["ROW_HALF"] == 0]

alloc_slope = (
    train_half.groupby("ALLOCATION")
    .apply(lambda g: g["RET_1"].cov(g["target"]) / g["RET_1"].var())
)

# apply to the untouched half
valid = df[df["ROW_HALF"] == 1].copy()
valid["ALLOC_SLOPE"] = valid["ALLOCATION"].map(alloc_slope)

print(valid[["ALLOC_SLOPE", "target"]].corr())

             ALLOC_SLOPE    target
ALLOC_SLOPE     1.000000 -0.004691
target         -0.004691  1.000000


In [45]:
df["RET1_CS_DECILE"] = (
    df.groupby("TS")["RET_1"]
      .rank(pct=True)
      .mul(10)
      .clip(upper=10)
      .astype(int)
)

print(
    df.groupby("RET1_CS_DECILE")["target"]
      .agg(["mean", "count"])
)

                    mean  count
RET1_CS_DECILE                 
0              -0.000326  51133
1              -0.000153  53206
2              -0.000098  51976
3              -0.000035  53205
4               0.000023  51740
5               0.000051  53379
6               0.000092  53482
7               0.000170  51699
8               0.000227  53482
9               0.000365  51249
10              0.000639   2522


In [48]:
ret_std = total_data[[f"RET_{i}" for i in range(1,21)]].std(axis=1)

total_data["RET1_VOL_NORM"] = total_data["RET_1"] / (ret_std + 1e-6)

print(total_data[["RET1_VOL_NORM", "target"]].corr())

               RET1_VOL_NORM    target
RET1_VOL_NORM       1.000000  0.070719
target              0.070719  1.000000


In [49]:
total_data["RET1_ABS"] = total_data["RET_1"].abs()
total_data["RET1_VOL_DECILE"] = pd.qcut(total_data["RET1_ABS"], 10, labels=False)

print(
    total_data.groupby("RET1_VOL_DECILE")
    .apply(lambda x: x["RET_1"].corr(x["target"]))
)

RET1_VOL_DECILE
0    0.005926
1    0.010707
2    0.017963
3    0.017951
4    0.037944
5    0.045977
6    0.049903
7    0.067870
8    0.078540
9    0.147184
dtype: float64


In [50]:
total_data["RET1_SQ"] = total_data["RET_1"] ** 2

print(total_data[["RET1_SQ", "target"]].corr())

          RET1_SQ    target
RET1_SQ  1.000000  0.006893
target   0.006893  1.000000


In [51]:
total_data["RET1_MAG_SIGN"] = total_data["RET_1"] * total_data["RET_1"].abs()

print(total_data[["RET1_MAG_SIGN", "target"]].corr())

               RET1_MAG_SIGN    target
RET1_MAG_SIGN       1.000000  0.056025
target              0.056025  1.000000


In [52]:
for q in [0.90, 0.95, 0.99]:
    cap = total_data["RET_1"].abs().quantile(q)
    f = total_data["RET_1"].clip(-cap, cap)
    print(q, f.corr(total_data["target"]))

0.9 0.07285077516018397
0.95 0.07565285271013863
0.99 0.0804893994126774


In [53]:
total_data["RET1_POS"] = total_data["RET_1"].clip(lower=0)
total_data["RET1_NEG"] = total_data["RET_1"].clip(upper=0)

print(total_data[["RET1_POS", "RET1_NEG", "target"]].corr()["target"])

RET1_POS    0.073023
RET1_NEG    0.064676
target      1.000000
Name: target, dtype: float64


In [54]:
med = total_data.groupby("TS")["RET_1"].transform("median")

total_data["RET1_CS_MAG"] = (
    total_data["RET_1"] - med
).abs() * np.sign(total_data["RET_1"])

print(total_data[["RET1_CS_MAG", "target"]].corr())

             RET1_CS_MAG    target
RET1_CS_MAG     1.000000  0.082323
target          0.082323  1.000000


In [55]:
total_data["RET1_CS_DEV"] = (
    total_data["RET_1"] -
    total_data.groupby("TS")["RET_1"].transform("median")
)

total_data["RET1_INTERACT"] = total_data["RET_1"] * total_data["RET1_CS_DEV"]

print(total_data[["RET1_INTERACT", "target"]].corr())

               RET1_INTERACT    target
RET1_INTERACT       1.000000  0.005269
target              0.005269  1.000000


In [56]:
vol = total_data[[f"RET_{i}" for i in range(2, 21)]].std(axis=1)

total_data["RET1_X_VOL"] = total_data["RET_1"] * vol

print(total_data[["RET1_X_VOL", "target"]].corr())

            RET1_X_VOL    target
RET1_X_VOL    1.000000  0.075585
target        0.075585  1.000000


In [57]:
total_data["VOL_2_20"] = total_data[[f"RET_{i}" for i in range(2, 21)]].std(axis=1)

total_data["VOL_DECILE"] = pd.qcut(
    total_data["VOL_2_20"], 10, labels=False, duplicates="drop"
)

print(
    total_data.groupby("VOL_DECILE")
    .apply(lambda g: g["RET_1"].corr(g["target"]))
)

VOL_DECILE
0    0.071188
1    0.065210
2    0.078277
3    0.074888
4    0.077827
5    0.065016
6    0.077742
7    0.088096
8    0.094529
9    0.092280
dtype: float64


In [58]:
total_data["VOL_DECILE"] = pd.qcut(
    total_data["VOL_2_20"], 10, labels=False, duplicates="drop"
)

total_data["RET1_X_VOLDEC"] = (
    total_data["RET_1"] *
    (total_data["VOL_DECILE"] / 9)
)

print(total_data[["RET1_X_VOLDEC", "target"]].corr())

               RET1_X_VOLDEC    target
RET1_X_VOLDEC       1.000000  0.082056
target              0.082056  1.000000


In [59]:
total_data["VOL_DECILE"] = pd.qcut(
    total_data["VOL_2_20"], 10, labels=False, duplicates="drop"
)

print(total_data.groupby("VOL_DECILE")["target"].agg(["mean", "std", "count"]))

                mean       std  count
VOL_DECILE                           
0          -0.000010  0.001490  52708
1           0.000009  0.001843  52707
2           0.000033  0.002096  52707
3           0.000048  0.002278  52707
4           0.000033  0.002475  52708
5           0.000063  0.002717  52707
6           0.000040  0.003020  52707
7           0.000046  0.003379  52707
8           0.000008  0.003979  52707
9           0.000076  0.005603  52708


In [60]:
total_data["RET1_X_VOLRATIO"] = (
    total_data["RET_1"] *
    total_data["VOL_2_20"] /
    total_data["RET_1"].abs().median()
)

print(total_data[["RET1_X_VOLRATIO", "target"]].corr())

                 RET1_X_VOLRATIO    target
RET1_X_VOLRATIO         1.000000  0.075585
target                  0.075585  1.000000


In [61]:
total_data["RET1_MINUS_OWN_MEAN"] = (
    total_data["RET_1"] -
    total_data[[f"RET_{i}" for i in range(2, 21)]].mean(axis=1)
)

print(total_data[["RET1_MINUS_OWN_MEAN", "target"]].corr())

                     RET1_MINUS_OWN_MEAN    target
RET1_MINUS_OWN_MEAN             1.000000  0.080776
target                          0.080776  1.000000


In [62]:
total_data["RET1_X_MEAN3"] = (
    total_data["RET_1"] *
    total_data[["RET_2", "RET_3", "RET_4"]].mean(axis=1)
)

print(total_data[["RET1_X_MEAN3", "target"]].corr())

              RET1_X_MEAN3    target
RET1_X_MEAN3      1.000000  0.000545
target            0.000545  1.000000


In [63]:
total_data["RET1_CORRECT"] = (
    np.sign(total_data["RET_1"]) == np.sign(total_data["target"])
).astype(int)

alloc_hit = total_data.groupby("ALLOCATION")["RET1_CORRECT"].mean()

print(alloc_hit.describe())
print(alloc_hit.sort_values().head(10))
print(alloc_hit.sort_values().tail(10))

count    278.000000
mean       0.519848
std        0.015287
min        0.484205
25%        0.508755
50%        0.519916
75%        0.528433
max        0.578947
Name: RET1_CORRECT, dtype: float64
ALLOCATION
ALLOCATION_18     0.484205
ALLOCATION_144    0.487661
ALLOCATION_221    0.489545
ALLOCATION_48     0.489867
ALLOCATION_238    0.490880
ALLOCATION_57     0.491455
ALLOCATION_231    0.492489
ALLOCATION_105    0.494099
ALLOCATION_39     0.494465
ALLOCATION_62     0.495172
Name: RET1_CORRECT, dtype: float64
ALLOCATION
ALLOCATION_263    0.547170
ALLOCATION_03     0.547869
ALLOCATION_38     0.549456
ALLOCATION_83     0.549456
ALLOCATION_167    0.550664
ALLOCATION_201    0.551528
ALLOCATION_163    0.553459
ALLOCATION_277    0.556953
ALLOCATION_46     0.578947
ALLOCATION_14     0.578947
Name: RET1_CORRECT, dtype: float64


In [64]:
total_data["SPLIT"] = total_data.groupby("ALLOCATION").cumcount() % 2

hit_by_alloc = (
    total_data.groupby(["ALLOCATION", "SPLIT"])["RET1_CORRECT"]
    .mean()
    .unstack()
)

print(hit_by_alloc.corr())

SPLIT         0         1
SPLIT                    
0      1.000000  0.167298
1      0.167298  1.000000


In [65]:
print(
    total_data.groupby("GROUP")["RET1_CORRECT"]
    .agg(["mean", "count"])
)

           mean   count
GROUP                  
1      0.523111  100170
2      0.521864  134928
3      0.514650  168913
4      0.518162  123062


In [66]:
total_data["RET1_WRONG"] = (
    np.sign(total_data["RET_1"]) != np.sign(total_data["target"])
).astype(int)

total_data["RET_PATH_MEAN"] = total_data[
    [f"RET_{i}" for i in range(2, 21)]
].mean(axis=1)

print(
    total_data.groupby("RET1_WRONG")["RET_PATH_MEAN"]
    .agg(["mean", "std", "count"])
)

                mean      std   count
RET1_WRONG                           
0           0.000029  0.00073  273511
1           0.000026  0.00072  253562


In [67]:
alloc_hit = total_data.groupby("ALLOCATION")["RET1_CORRECT"].transform("mean")
group_hit = total_data.groupby("GROUP")["RET1_CORRECT"].transform("mean")

total_data["ALLOC_HIT_REL_GROUP"] = alloc_hit - group_hit

print(total_data[["ALLOC_HIT_REL_GROUP", "target"]].corr())

                     ALLOC_HIT_REL_GROUP    target
ALLOC_HIT_REL_GROUP             1.000000  0.002613
target                          0.002613  1.000000


In [68]:
total_data["RET1_RESID"] = (
    total_data["target"] - total_data["RET_1"]
)

print(
    total_data[[f"RET_{i}" for i in range(2, 21)] + ["RET1_RESID"]]
    .corr()["RET1_RESID"]
    .sort_values()
)

RET_2        -0.056384
RET_10       -0.014241
RET_17       -0.012292
RET_4        -0.009350
RET_18       -0.007715
RET_15       -0.002781
RET_9        -0.001597
RET_3        -0.000279
RET_12       -0.000038
RET_14        0.000018
RET_20        0.000453
RET_8         0.001180
RET_11        0.001938
RET_13        0.002286
RET_16        0.003935
RET_19        0.006186
RET_7         0.007820
RET_6         0.013223
RET_5         0.016054
RET1_RESID    1.000000
Name: RET1_RESID, dtype: float64


In [69]:
total_data["RET2_RESID_SIGNAL"] = (
    total_data["RET_2"] -
    total_data.groupby("TS")["RET_2"].transform("median")
)

print(
    total_data[["RET2_RESID_SIGNAL", "RET1_RESID"]].corr()
)

                   RET2_RESID_SIGNAL  RET1_RESID
RET2_RESID_SIGNAL           1.000000   -0.055247
RET1_RESID                 -0.055247    1.000000


In [70]:
for k in [0.25, 0.5, 1.0]:
    pred = total_data["RET_1"] + k * total_data["RET2_RESID_SIGNAL"]
    acc = (
        np.sign(pred) == np.sign(total_data["target"])
    ).mean()
    print(k, acc)

print("RET1:", (
    np.sign(total_data["RET_1"]) == np.sign(total_data["target"])
).mean())

0.25 0.5170612040457394
0.5 0.5147693014060671
1.0 0.510483367579064
RET1: 0.5189243235756716


In [71]:
total_data["RET1_SIGN"] = np.sign(total_data["RET_1"])

print(
    total_data.groupby("RET1_SIGN")
    .apply(lambda g: g["RET_2"].corr(g["RET1_RESID"]))
)

RET1_SIGN
-1.0   -0.037621
 1.0   -0.053338
dtype: float64


In [72]:
total_data["RET1_ABS_DECILE"] = pd.qcut(
    total_data["RET_1"].abs(), 10, labels=False, duplicates="drop"
)

print(
    total_data.groupby("RET1_ABS_DECILE")
    .apply(lambda g: g["RET_2"].corr(g["RET1_RESID"]))
)

RET1_ABS_DECILE
0   -0.009837
1   -0.005039
2   -0.015472
3   -0.016230
4   -0.027101
5   -0.013528
6   -0.025289
7   -0.031055
8   -0.040119
9   -0.126048
dtype: float64


In [73]:
g = total_data[total_data["RET1_ABS_DECILE"] == 9].copy()

g["RET2_SIGNAL"] = (
    g["RET_2"] - g.groupby("TS")["RET_2"].transform("median")
)

print(g.groupby("RET1_WRONG")["RET2_SIGNAL"].agg(["mean", "median", "std", "count"]))
print("corr:", g["RET2_SIGNAL"].corr(g["RET1_RESID"]))

                mean  median       std  count
RET1_WRONG                                   
0          -0.000057     0.0  0.004209  28765
1           0.000061     0.0  0.004281  23943
corr: -0.0939652510156275


In [75]:
g = total_data[total_data["RET1_ABS_DECILE"] == 9].copy()

g["RET2_SIGNAL"] = g["RET_2"] - g.groupby("TS")["RET_2"].transform("median")

for k in [0.25, 0.5, 1.0]:
    pred = np.sign(g["RET_1"] - k * g["RET2_SIGNAL"])
    print(k, (pred == np.sign(g["target"])).mean())

print("raw:", (np.sign(g["RET_1"]) == np.sign(g["target"])).mean())

0.25 0.5458374440312665
0.5 0.5455149123472718
1.0 0.5424224026713212
raw: 0.5457425817712681


In [76]:
g["RET2_ABS"] = g["RET_2"].abs()

print(
    g.groupby(pd.qcut(g["RET2_ABS"], 10, duplicates="drop"))
     .apply(lambda x: pd.Series({
         "wrong_rate": x["RET1_WRONG"].mean(),
         "resid_corr": x["RET2_SIGNAL"].corr(x["RET1_RESID"]),
     }))
)

                                wrong_rate  resid_corr
RET2_ABS                                              
(-0.000999999999802, 0.000454]    0.441852    0.015684
(0.000454, 0.000913]              0.436160    0.016854
(0.000913, 0.0014]                0.453804    0.002656
(0.0014, 0.00192]                 0.464137   -0.000642
(0.00192, 0.00252]                0.458736    0.011976
(0.00252, 0.00321]                0.443369   -0.045988
(0.00321, 0.00411]                0.469639   -0.078698
(0.00411, 0.00535]                0.461203   -0.109889
(0.00535, 0.00747]                0.453235   -0.118271
(0.00747, 0.0494]                 0.460444   -0.192745


In [78]:
for q in [0.7, 0.8, 0.9, 0.95]:
    cutoff = g["RET2_ABS"].quantile(q)
    h = g[g["RET2_ABS"] >= cutoff]

    pred = np.sign(h["RET_1"] - 0.25 * h["RET2_SIGNAL"])

    print(
        q,
        "n=", len(h),
        "corrected=", (pred == np.sign(h["target"])).mean(),
        "raw=", (np.sign(h["RET_1"]) == np.sign(h["target"])).mean()
    )

0.7 n= 15813 corrected= 0.5419591475368368 raw= 0.5417061911085815
0.8 n= 10542 corrected= 0.5434452665528363 raw= 0.5431606905710491
0.9 n= 5271 corrected= 0.5399354961107949 raw= 0.5395560614684121
0.95 n= 2636 corrected= 0.5349013657056145 raw= 0.5341426403641881


In [80]:
h = g[g["RET2_ABS"] >= g["RET2_ABS"].quantile(0.8)]

for k in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    pred = np.sign(h["RET_1"] - k * h["RET2_SIGNAL"])
    print(k, (pred == np.sign(h["target"])).mean())

print("raw:", (np.sign(h["RET_1"]) == np.sign(h["target"])).mean())

0.05 0.5431606905710491
0.1 0.5431606905710491
0.15 0.5431606905710491
0.2 0.5433504078922405
0.25 0.5434452665528363
0.3 0.5442989944981976
0.4 0.5446784291405805
raw: 0.5431606905710491


In [82]:
h = g[g["RET2_ABS"] >= g["RET2_ABS"].quantile(0.8)].copy()

for k in [0.2, 0.3, 0.4]:
    h["pred"] = np.sign(h["RET_1"] - k * h["RET2_SIGNAL"])
    flipped = h["pred"] != np.sign(h["RET_1"])

    print(
        "k", k,
        "flips", flipped.mean(),
        "flip accuracy", (
            h.loc[flipped, "pred"] == np.sign(h.loc[flipped, "target"])
        ).mean(),
        "n", flipped.sum()
    )

k 0.2 flips 0.00018971732119142478 flip accuracy 1.0 n 2
k 0.3 flips 0.004553215708594195 flip accuracy 0.625 n 48
k 0.4 flips 0.014418516410548284 flip accuracy 0.5526315789473685 n 152


In [83]:
h["RET2_SIG_DECILE"] = pd.qcut(
    h["RET2_SIGNAL"], 10, labels=False, duplicates="drop"
)

print(
    h.groupby("RET2_SIG_DECILE")["RET1_WRONG"]
     .agg(["mean", "count"])
)

                     mean  count
RET2_SIG_DECILE                 
0                0.462559   1055
1                0.462049   1054
2                0.444023   1054
3                0.387097   1054
4                0.450664   1054
5                0.441176   1054
6                0.487666   1054
7                0.497154   1054
8                0.458254   1054
9                0.477725   1055


In [84]:
g["OPPOSITE"] = np.sign(g["RET_1"]) != np.sign(g["RET_2"])

print(
    g.groupby("OPPOSITE")
     .apply(lambda x: pd.Series({
         "wrong_rate": x["RET1_WRONG"].mean(),
         "resid_corr": x["RET_2"].corr(x["RET1_RESID"]),
         "n": len(x)
     }))
)

          wrong_rate  resid_corr        n
OPPOSITE                                 
False       0.456505   -0.621317  28946.0
True        0.451519    0.553065  23762.0


In [86]:
for k in [0.05, 0.10, 0.20, 0.30, 0.40]:
    # RET_2 should be used in opposite directions depending on
    # whether RET_1 and RET_2 agree
    correction = np.where(
        np.sign(g["RET_1"]) == np.sign(g["RET_2"]),
        -g["RET_2"],
        g["RET_2"]
    )

    pred = np.sign(g["RET_1"] + k * correction)

    print(k, (pred == np.sign(g["target"])).mean())

print("raw:", (np.sign(g["RET_1"]) == np.sign(g["target"])).mean())

0.05 0.5457425817712681
0.1 0.5457425817712681
0.2 0.5457805266752676
0.3 0.5457994991272672
0.4 0.5452113531152766
raw: 0.5457425817712681


In [87]:
g["RET1_RET2_PRODUCT"] = g["RET_1"] * g["RET_2"]

print("product corr:", g["RET1_RET2_PRODUCT"].corr(g["RET1_RESID"]))

print(
    g.groupby(pd.qcut(g["RET1_RET2_PRODUCT"], 10, duplicates="drop"))
     .apply(lambda x: pd.Series({
         "resid_mean": x["RET1_RESID"].mean(),
         "wrong_rate": x["RET1_WRONG"].mean()
     }))
)

product corr: -0.05596523898914123
                        resid_mean  wrong_rate
RET1_RET2_PRODUCT                             
(-0.00198, -3.15e-05]     0.000013    0.439006
(-3.15e-05, -1.68e-05]    0.000323    0.458926
(-1.68e-05, -8.99e-06]    0.000101    0.461962
(-8.99e-06, -2.82e-06]    0.000093    0.446679
(-2.82e-06, 2.7e-06]      0.000080    0.443369
(2.7e-06, 8.28e-06]      -0.000211    0.450009
(8.28e-06, 1.49e-05]     -0.000427    0.457875
(1.49e-05, 2.4e-05]      -0.000304    0.450958
(2.4e-05, 4.17e-05]      -0.000488    0.468792
(4.17e-05, 0.00119]      -0.000944    0.464997


In [88]:
eps = 1e-6

g["RET2_RET1_RATIO"] = g["RET_2"] / (g["RET_1"].abs() + eps)

print("corr:", g["RET2_RET1_RATIO"].corr(g["RET1_RESID"]))

print(
    g.groupby(pd.qcut(
        g["RET2_RET1_RATIO"].clip(-10, 10),
        10, duplicates="drop"
    ))["RET1_WRONG"].mean()
)

corr: -0.09054044845641615
RET2_RET1_RATIO
(-7.297000000000001, -0.787]    0.447353
(-0.787, -0.469]                0.455511
(-0.469, -0.269]                0.470309
(-0.269, -0.112]                0.451423
(-0.112, 0.0275]                0.444697
(0.0275, 0.166]                 0.433694
(0.166, 0.319]                  0.458444
(0.319, 0.513]                  0.440524
(0.513, 0.827]                  0.460634
(0.827, 7.632]                  0.479985
Name: RET1_WRONG, dtype: float64


In [89]:
# How unusual is today's RET_1 relative to its own prior 19 returns?
hist = total_data[[f"RET_{i}" for i in range(2, 21)]]

total_data["RET1_OWN_Z"] = (
    total_data["RET_1"] - hist.mean(axis=1)
) / (hist.std(axis=1) + 1e-6)

# Test whether this actually identifies failures
total_data.groupby(
    pd.qcut(total_data["RET1_OWN_Z"].abs(), 10, labels=False)
)["RET1_WRONG"].mean()

RET1_OWN_Z
0    0.489470
1    0.493976
2    0.490865
3    0.487222
4    0.483798
5    0.481397
6    0.477906
7    0.476730
8    0.473011
9    0.456382
Name: RET1_WRONG, dtype: float64

In [90]:
total_data["RET1_CS_ABS"] = (
    total_data["RET_1"] -
    total_data.groupby("TS")["RET_1"].transform("median")
).abs()

total_data.groupby(
    pd.qcut(total_data["RET1_CS_ABS"], 10, labels=False)
)["RET1_WRONG"].mean()

RET1_CS_ABS
0    0.496016
1    0.488569
2    0.494033
3    0.486577
4    0.483323
5    0.483826
6    0.480012
7    0.473182
8    0.470032
9    0.455187
Name: RET1_WRONG, dtype: float64

In [91]:
total_data["RET1_VOL_CONFIRM"] = (
    np.sign(total_data["RET_1"]) *
    total_data["SIGNED_VOLUME_1"]
)

total_data.groupby(
    pd.qcut(total_data["RET1_VOL_CONFIRM"], 10, labels=False)
)["RET1_WRONG"].mean()


RET1_VOL_CONFIRM
0.0    0.479544
1.0    0.482625
2.0    0.497492
3.0    0.487497
4.0    0.496597
5.0    0.485024
6.0    0.469012
7.0    0.478504
8.0    0.491725
9.0    0.489360
Name: RET1_WRONG, dtype: float64

In [93]:
total_data["RET1_VOL_CONTRADICT"] = (
    np.sign(total_data["RET_1"]) !=
    np.sign(total_data["SIGNED_VOLUME_1"])
).astype("int8")

print(total_data.groupby("RET1_VOL_CONTRADICT")["RET1_WRONG"].agg(["mean", "count"]))

                         mean   count
RET1_VOL_CONTRADICT                  
0                    0.482635   71292
1                    0.480832  455781


In [95]:
total_data["RET1_REVERSAL"] = (
    np.sign(total_data["RET_1"]) != np.sign(total_data["RET_2"])
).astype("int8")

print(total_data.groupby("RET1_REVERSAL")["RET1_WRONG"].agg(["mean", "count"]))

                   mean   count
RET1_REVERSAL                  
0              0.483996  271126
1              0.477982  255947


In [97]:
x = total_data[
    np.sign(total_data["RET_1"]) != np.sign(total_data["RET_2"])
].copy()

x["RET2_MAG_DECILE"] = pd.qcut(
    x["RET_2"].abs(), 10, labels=False, duplicates="drop"
)

print(x.groupby("RET2_MAG_DECILE")["RET1_WRONG"].agg(["mean", "count"]))

                     mean  count
RET2_MAG_DECILE                 
0                0.481149  25595
1                0.476890  25595
2                0.480738  25594
3                0.481188  25595
4                0.476538  25595
5                0.481167  25594
6                0.484743  25595
7                0.474017  25594
8                0.475249  25595
9                0.468138  25595


In [98]:
x = total_data[
    np.sign(total_data["RET_1"]) != np.sign(total_data["RET_2"])
].copy()

x["REVERSAL_RATIO"] = (
    x["RET_1"].abs() / (x["RET_2"].abs() + 1e-6)
)

x["RATIO_DECILE"] = pd.qcut(
    x["REVERSAL_RATIO"], 10, labels=False, duplicates="drop"
)

x.groupby("RATIO_DECILE")["RET1_WRONG"].agg(["mean", "count"])

,mean,count
RATIO_DECILE,,
0,0.491815,25595
1,0.485954,25595
2,0.483043,25594
3,0.475835,25595
4,0.479937,25595
5,0.477729,25594
6,0.473764,25595
7,0.474642,25594
8,0.474038,25595
